# Hospital Fleet Core Logic Notebook

This notebook replicates the scheduler prediction and assignment logic without ROS dependencies, for quick testing.

In [ ]:
import networkx as nx
from scipy.optimize import linear_sum_assignment

def build_hospital_map():
    G=nx.Graph()
    edges=[('Lobby','NurseStation'),('NurseStation','ER'),('NurseStation','WardA'),('NurseStation','WardB'),('ER','ICU'),('WardA','Lab'),('WardB','Pharmacy'),('Lab','Radiology'),('Pharmacy','Cafeteria'),('Cafeteria','Lobby')]
    G.add_edges_from(edges, weight=1)
    return G

def predict_time(distance, congestion, base=5.0, d_coeff=2.0, c_coeff=4.0):
    return base + d_coeff*distance + c_coeff*congestion

G=build_hospital_map()
robots={'robot_1':'Lobby','robot_2':'NurseStation','robot_3':'WardA'}
tasks=[{'task_id':'t1','from':'Pharmacy','to':'WardB'},{'task_id':'t2','from':'Lab','to':'Radiology'},{'task_id':'t3','from':'ER','to':'ICU'}]
matrix=[]
for r,pos in robots.items():
    row=[]
    for t in tasks:
        d1=nx.shortest_path_length(G,pos,t['from'])
        d2=nx.shortest_path_length(G,t['from'],t['to'])
        tot=d1+d2
        cong=max(0,tot-2)
        row.append(predict_time(tot,cong))
    matrix.append(row)
row_ind,col_ind=linear_sum_assignment(matrix)
print('Assignments:')
for r,c in zip(row_ind,col_ind):
    print(list(robots.keys())[r], '->', tasks[c]['task_id'])